# Integrating photometry & spectroscopy in StarKit #

Filter curves come from the SVO Filter Profile Service (http://svo2.cab.inta-csic.es/theory/fps/). See [Synthetic photometry](../photometry.rst).

In [1]:
from starkit.gridkit import load_grid
from starkit.base.operations.imager import Photometry
from starkit.filters import Filter
from starkit.fitkit.likelihoods import PhotometryColorLikelihood

In [2]:
GRID_PATH = 'phoenix_t4000_10000_w3000_9000_r3000.h5'

In [3]:
grid = load_grid(GRID_PATH)

[starkit.gridkit.base][INFO   ]  Reading index (base.py:300)


[starkit.gridkit.base][INFO   ]  Discovered columns teff, logg, mh (base.py:303)


[starkit.gridkit.base][INFO   ]  Reading Fluxes (base.py:306)


[starkit.gridkit.base][INFO   ]  Fluxes shape (688, 13184) (base.py:309)


[py.warnings         ][WARNING]  /private/tmp/claude-503/-Users-tdo-python-starkit/ea1e8202-1dc3-4e2e-b739-34a668b07e46/scratchpad/venv/lib/python3.12/site-packages/astropy/units/format/generic.py:506: UnitsWarning: 'erg/s/angstrom' contains multiple slashes, which is discouraged by the FITS standard
  warnings.warn(
 (warnings.py:112)


[starkit.gridkit.base][INFO   ]  Initializing spec grid (base.py:343)


[starkit.gridkit.base][WARNING]  **** NO WAVELENGTH TYPE SET DEFAULTING TO GRID (vacuum) ****

 (base.py:108)


The generic Johnson curves. SVO lists them as energy responses, which `Filter.from_svo` reads from the `DetectorType` of each curve.

In [4]:
johnson = [Filter.from_svo(filter_id) for filter_id in
           ['Generic/Johnson.U', 'Generic/Johnson.B', 'Generic/Johnson.V']]
johnson

[<Filter Generic/Johnson.U (energy), 3000 to 4200 Angstrom>,
 <Filter Generic/Johnson.B (energy), 3700 to 5600 Angstrom>,
 <Filter Generic/Johnson.V (energy), 4600 to 7400 Angstrom>]

In [5]:
phot_operation = Photometry.from_grid(grid, filter_set=johnson, mag_type='vega')

Example observed magnitudes and uncertainties in U, B and V, in the order of the filters.

In [6]:
observed = [17.6418, 17.2215, 17.0349]
observed_uncertainty = [0.0346, 0.0532, 0.0659]

The model computes magnitudes from the flux stored in the grid, without a distance. Check what the fluxes of the grid are with `check_flux_scale` (see [Grid flux units](../io/flux_units.rst)). For a grid of surface flux, these are magnitudes of the stellar surface: their differences (colors) can be compared with observed colors, but comparing the magnitudes themselves with observations requires scaling the flux by $(R/d)^2$, with $R$ the stellar radius and $d$ the distance.

In [7]:
phot_model = grid | phot_operation

In [8]:
phot_model.teff_0 = 9940
phot_model.logg_0 = 4.33
phot_model.mh_0 = -0.5

In [9]:
phot_model()

array([-40.62362143, -40.53871374, -40.55074621])

The likelihood of the observed colors, U - B and B - V, which do not depend on the flux scale of the grid.

In [10]:
likelihood = phot_model | PhotometryColorLikelihood(observed, observed_uncertainty)
likelihood()

-33.81148135532842